# HyperNet strict-FP32 GPU portability pilot

**This is not a new accuracy or discovery experiment.** Eight previously used development events and four existing positive reference events, with four authentic duration PNGs each, check that the locked model transfers to this runtime.

The encoder uses 42,480 parameters and mathematical-only pretraining. Each target has one positive reference event plus separate negative fitting/calibration data; the stored matcher and thresholds remain fixed. No training, fitting, downloads, or reserved-data access occurs.

Attach the private strict-FP32 pilot dataset, select up to two GPUs, and keep Internet off. CPU-only environments verify saved artifacts without neural inference.

This is a separately versioned numerical migration: CPU FP32 descriptors, unchanged encoder and exact original readout fitting/calibration policy. The historical TF32 release and failed portability pilot are retained.


In [ ]:
from pathlib import Path
import json, sys, subprocess, zipfile, stat, time

input_root = Path('/kaggle/input')
work_root = Path('/kaggle/working')
work_root.mkdir(parents=True, exist_ok=True)
candidates = []
for path in input_root.rglob('manifest.json'):
    try:
        metadata = json.loads(path.read_text())
        if metadata.get('deployment_version') == 'hypernet_gs_s45029_joint4_strict_fp32_v1' and metadata.get('format_version') == 1 and metadata.get('query_count') == 8 and metadata.get('support_count') == 4 and metadata.get('encoder_parameters') == 42480:
            candidates.append(path.parent)
    except (ValueError, OSError):
        pass
if not candidates:
    archives = list(input_root.rglob('hypernet_gs_fp32_kaggle_pilot_v1.zip'))
    assert len(archives) == 1, 'Attach exactly one private pilot dataset or its named ZIP.'
    unpacked = work_root / 'hypernet_gs_fp32_pilot_payload'
    unpacked.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.infolist():
            target = (unpacked / entry.filename).resolve()
            assert target.is_relative_to(unpacked.resolve()), 'Unsafe archive path'
            assert not stat.S_ISLNK(entry.external_attr >> 16), 'Archive symlinks are not allowed'
        archive.extractall(unpacked)
    candidates = [unpacked]
assert len(candidates) == 1, 'Ambiguous pilot datasets; keep one attached.'
PILOT_ROOT = candidates[0]
RUN_ROOT = work_root / ('hypernet_fp32_pilot_' + time.strftime('%Y%m%d_%H%M%S'))
print('Payload:', PILOT_ROOT)
print('Outputs:', RUN_ROOT)


In [ ]:
subprocess.run([sys.executable, str(PILOT_ROOT / 'run_pilot.py'),
    '--root', str(PILOT_ROOT), '--verify-only',
    '--output', str(RUN_ROOT / 'verify')], check=True)


In [ ]:
# At most two independent GPU workers. The fixed fixture is split by event;
# supports remain the same four events, and no weights or thresholds are fit.
subprocess.run([sys.executable, str(PILOT_ROOT / 'run_pilot.py'),
    '--root', str(PILOT_ROOT), '--max-gpus', '2', '--repeats', '3',
    '--output', str(RUN_ROOT / 'gpu')], check=True)
result = json.loads((RUN_ROOT / 'gpu' / 'pilot_result.json').read_text())
print(json.dumps({key: result[key] for key in ('status', 'passed', 'gpu_jobs')}, indent=2))


## Reading the output

A passing result establishes only this fixture's hash, preprocessing, descriptor, score, and decision portability. It is not an accuracy estimate, low-false-alarm validation, rare-target result, or a five-hour archive ETA. Inspect the timing stages and any drift failures; never relax the predeclared tolerance to force a pass. Download the result JSON and per-GPU feature/score files from the notebook output.
